# Fighting Overfitting

Overfitting happens when a model memorizes training data but performs poorly on new data.

```text
Training loss:   keeps decreasing
Validation loss: decreases, then starts increasing  <- overfitting
```

Common reasons:

- Too little training data
- A model with more capacity than the task needs
- Weights growing too large
- Training for too many epochs

## Common Solutions

```mermaid
flowchart LR
    O[Overfitting] --> S[Smaller model]
    O --> D[Dropout]
    O --> W[Weight decay]
    O --> M[More data]
    O --> A[Data augmentation]
    O --> E[Stop training earlier]
```

## Reduce Model Size

More neurons and layers mean more parameters that can memorize the training data. Start with a smaller model when the dataset is small.

In [ ]:
import torch
from torch import nn

large_model = nn.Sequential(
    nn.Linear(7, 128),
    nn.ReLU(),
    nn.Linear(128, 128),
    nn.ReLU(),
    nn.Linear(128, 3),
)

small_model = nn.Sequential(
    nn.Linear(7, 16),
    nn.ReLU(),
    nn.Linear(16, 3),
)

large_parameter_count = sum(p.numel() for p in large_model.parameters())
small_parameter_count = sum(p.numel() for p in small_model.parameters())

print(f"Large model parameters: {large_parameter_count:,}")
print(f"Small model parameters: {small_parameter_count:,}")

## Dropout

Dropout randomly turns off a fraction of values during training. This prevents the model from depending too heavily on specific neurons.

```python
nn.Dropout(p=0.5)  # each value has a 50% chance of being dropped
```

Dropout is normally placed after an activation function.

In [ ]:
torch.manual_seed(42)

dropout = nn.Dropout(p=0.5)
example_values = torch.ones(1, 8)

dropout.train()  # enable random dropping
training_output_1 = dropout(example_values)
training_output_2 = dropout(example_values)

dropout.eval()  # disable dropping for stable predictions
evaluation_output_1 = dropout(example_values)
evaluation_output_2 = dropout(example_values)

print(f"Training output 1:   {training_output_1}")
print(f"Training output 2:   {training_output_2}")
print(f"Evaluation output 1: {evaluation_output_1}")
print(f"Evaluation output 2: {evaluation_output_2}")

During training, dropped values become `0`. The remaining values are scaled so the expected output stays consistent. With `p=0.5`, kept values become `2`.

During evaluation, nothing is dropped, so repeated predictions are stable. This is why `model.train()` and `model.eval()` matter.

### Dropout Inside a Model

In [ ]:
model = nn.Sequential(
    nn.Linear(7, 16),
    nn.ReLU(),
    nn.Dropout(p=0.5),  # randomly drops hidden values during training only
    nn.Linear(16, 3),
)

## Weight Decay

Weight decay discourages weights from becoming too large. Add it when creating the optimizer:

```text
gradient used for update = loss gradient + weight_decay * parameter
parameter = parameter - learning_rate * gradient used for update
```

A larger `weight_decay` creates a stronger push toward smaller weights, but too much can cause underfitting.

In [ ]:
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.1,
    weight_decay=0.001,  # penalize large parameter values during updates
)

print(optimizer)

## Data Augmentation

Data augmentation creates modified training examples without collecting new data.

```mermaid
flowchart LR
    I[Original training image] --> R[Rotate]
    I --> F[Flip]
    I --> C[Crop or scale]
    R --> D[More training examples]
    F --> D
    C --> D
```

Apply random augmentation only to the **training set**. Validation and test data should stay consistent so they measure the model fairly.

## Training and Validation

Dropout automatically follows the model's mode. Weight decay is applied by `optimizer.step()`.

In [ ]:
criterion = nn.CrossEntropyLoss()

# TRAINING
model.train()  # enable Dropout
optimizer.zero_grad()  # clear gradients from the previous batch

training_inputs = torch.randn(4, 7)
training_labels = torch.tensor([0, 1, 2, 1])
scores = model(training_inputs)
loss = criterion(scores, training_labels)
loss.backward()   # calculate gradients from the current batch
optimizer.step()  # update parameters and apply weight decay

# VALIDATION
model.eval()  # disable Dropout
with torch.no_grad():  # validation does not calculate gradients or update parameters
    validation_scores = model(training_inputs)
    validation_loss = criterion(validation_scores, training_labels)

print(f"Training loss: {loss.item():.4f}")
print(f"Validation-style loss: {validation_loss.item():.4f}")

## Quick Look

| Method | What it does | Where it acts |
|---|---|---|
| Smaller model | Reduces capacity | Architecture |
| Dropout | Randomly drops hidden values | Training forward pass |
| Weight decay | Encourages smaller weights | Optimizer update |
| More data | Adds real examples | Dataset |
| Data augmentation | Creates modified examples | Training dataset |
| Early stopping | Stops before validation worsens | Training schedule |

```python
model = nn.Sequential(
    nn.Linear(7, 16),
    nn.ReLU(),
    nn.Dropout(p=0.5),
    nn.Linear(16, 3),
)

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.1,
    weight_decay=0.001,
)
```